In [6]:
import os
from IPython.display import display, Image
from ipywidgets import interact, IntSlider

def display_images_in_folder(folder_path):
    """
    Displays all images in a folder as scrollable output in a Jupyter Notebook.

    Args:
        folder_path (str): The path to the folder containing the images.
    """

    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]
    image_paths = [os.path.join(folder_path, f) for f in image_files]

    if not image_paths:
        print(f"No images found in folder: {folder_path}")
        return

    def show_image(index):
        if 0 <= index < len(image_paths):
            display(Image(filename=image_paths[index]))
        else:
            print("Index out of range.")

    interact(show_image, index=IntSlider(min=0, max=len(image_paths) - 1, step=1, description='Image Index'))

# Example usage
folder_path = 'inference_results' 
display_images_in_folder(folder_path)

interactive(children=(IntSlider(value=0, description='Image Index', max=7), Output()), _dom_classes=('widget-i…

In [2]:
import json
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import ipywidgets as widgets
from ipywidgets import interact, IntSlider
import os

# Load the JSON data (run this cell once)
try:
    with open('yolo_10shot.json', 'r') as f:
        coco_data = json.load(f)
except FileNotFoundError:
    print("Error: distilled_labels.json not found. Please make sure the file is in the correct directory.")
    coco_data = None

if coco_data:
    images = coco_data['images']
    annotations = coco_data['annotations']

    # Create a dictionary to quickly access annotations by image_id
    image_annotations = {}
    for ann in annotations:
        image_id = ann['image_id']
        if image_id not in image_annotations:
            image_annotations[image_id] = []
        image_annotations[image_id].append(ann)

    def display_image_with_annotations(image_index):
        """
        Loads an image, draws its annotations, and displays it.
        """
        if image_index < 0 or image_index >= min(50, len(images)):
            print("Invalid image index.")
            return

        image_info = images[image_index]
        image_id = image_info['id']
        file_name = image_info['file_name']

        image_path = os.path.join('datasets/trucks_full_dataset/train', file_name)
        


        # Check if the image file exists before attempting to open
        if not os.path.exists(image_path):
            print(f"Image file not found: {image_path}")
            # Create a blank image or display a placeholder if the image is not found
            img = Image.new('RGB', (300, 200), color = (255, 255, 255))
            draw = ImageDraw.Draw(img)
            text = "Image not found"
            # Get text size using a default font
            try:
                text_width, text_height = draw.textsize(text)
            except AttributeError:
                # Fallback for newer Pillow versions where textsize is removed
                from PIL import ImageFont
                try:
                    font = ImageFont.load_default()
                    text_width, text_height = draw.textsize(text, font=font)
                except Exception:
                    text_width, text_height = 100, 20 # Approximate size if font loading fails

            draw.text(((300 - text_width) // 2, (200 - text_height) // 2), text, fill=(0,0,0))
            plt.figure(figsize=(6, 4))
            plt.imshow(img)
            plt.title(f"Image ID: {image_id}, File: {file_name} (Not found)")
            plt.axis('off')
            plt.show()
            return

        # Load the image file
        try:
            img = Image.open(image_path).convert('RGB')
            draw = ImageDraw.Draw(img)
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
            return

        # Get annotations for the current image
        annotations_for_image = image_annotations.get(image_id, [])

        # Draw the bounding boxes on the image
        for ann in annotations_for_image:
            bbox = ann['bbox'] # [x, y, width, height]
            # COCO bbox format is [x_min, y_min, width, height]
            x_min, y_min, width, height = bbox
            x_max = x_min + width
            y_max = y_min + height
            draw.rectangle([(x_min, y_min), (x_max, y_max)], outline="red", width=2)

        # Display the image
        plt.figure(figsize=(10, 10)) # Optional: Adjust figure size
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"Image ID: {image_id}, File: {file_name} ({len(annotations_for_image)} annotations)")
        plt.show()

    # Create a slider widget for the first 50 images
    image_slider = IntSlider(min=0, max=min(50, len(images)) - 1, step=1, description='Image Index:')

    # Use interact to link the slider to the display function
    interact(display_image_with_annotations, image_index=image_slider);
else:
    print("Could not load data. Please check the file 'distilled_labels.json'.")

interactive(children=(IntSlider(value=0, description='Image Index:', max=49), Output()), _dom_classes=('widget…